# RE-RL: Генерация данных для Theorem Proving (LeanNavigator подход)

**Полностью воспроизводимый пайплайн** для генерации training data из формальных доказательств Lean 4.

---

## Подход LeanNavigator — как это работает

> **Paper**: *"Generating Millions of Lean Theorems with Proofs by Exploring State Transition Graphs"*  
> David Yin, Jing Gao (Purdue University, February 2025)

### Ключевая идея

В Lean 4 доказательство теоремы — это **последовательность переходов между состояниями** (proof states). Каждое состояние содержит гипотезы и цель (goal), а **тактика** (tactic) — это команда, которая трансформирует одно состояние в другое. Когда все цели закрыты, мы достигаем состояния `ProofFinished` — доказательство найдено.

```
Формулировка теоремы (начальное состояние)
    │
    ├── tactic₁ → Новое состояние₁
    │       ├── tactic₃ → ProofFinished ✓
    │       └── tactic₄ → Новое состояние₃ → ...
    │
    ├── tactic₂ → Новое состояние₂
    │       └── tactic₅ → ProofFinished ✓
    │
    └── tactic₆ → Ошибка (тактика не применима) ✗
```

**LeanNavigator рассматривает это как граф**, где:
- **Вершины** = состояния доказательства (proof states)
- **Рёбра** = успешные применения тактик
- **Цель** = найти пути к `ProofFinished`

Любой путь от любого состояния к `ProofFinished` — это **новая теорема с доказательством**.

### Алгоритм: BFS (поиск в ширину) по графу состояний

```
Вход: теорема T из Mathlib4 (или другого Lean-репозитория)

1. ИНИЦИАЛИЗАЦИЯ:
   - Открываем T в LeanDojo (интерактивная среда Lean)
   - Получаем начальное состояние S₀ (goal теоремы)
   - Помещаем S₀ в приоритетную очередь Q (приоритет = глубина)

2. ЦИКЛ BFS (пока Q не пуста и не истёк лимит):
   a) Извлекаем состояние S с минимальным приоритетом из Q
   b) Генерируем кандидаты тактик для S:
      - Rule-based: базовые тактики (simp, ring, omega, rfl, ...)
      - Шаблоны: apply {h}, rw [{h}], cases {x}, induction {x}, ...
        где {h} и {x} подставляются из гипотез и переменных состояния S
      - [Опционально] Embedding retrieval (BERT + FAISS) для ускорения
   c) Для каждой тактики t:
      - Применяем t к S через LeanDojo → получаем результат
      - Если успех → новое состояние S':
        * Записываем training pair: (S, t, S')
        * Если S' = ProofFinished → отмечаем доказательство
        * Если S' новое → добавляем в Q
      - Если ошибка → пропускаем

3. ОБРАТНЫЙ ПРОХОД (backtrack distances):
   - Для каждого состояния считаем расстояние до ближайшего ProofFinished
   - distance_to_proof = 0 для прямых предшественников ProofFinished
   - distance_to_proof = 1 для их предшественников, и т.д.
   - Состояния дальше 8 шагов от ProofFinished отбрасываются

Выход: множество training pairs (state, tactic, next_state, distance_to_proof)
```

### Почему из 2 теорем получается 50+ training pairs?

BFS исследует **все применимые тактики** в каждом состоянии, не только ту, что ведёт к доказательству:

| Что записывается | Пример | Ценность |
|---|---|---|
| **Прямые доказательства** | `omega` закрывает цель сразу | Самые ценные (distance=0) |
| **Альтернативные пути** | `norm_cast` тоже закрывает цель | Модель учит несколько решений |
| **Промежуточные шаги** | `induction a` → два подцели | Учат стратегическому мышлению |
| **Ветви без proof** | `congr` → сложные подцели | Учат, какие тактики применимы |

### Масштаб оригинального LeanNavigator

| Метрика | Значение |
|---|---|
| Репозиторий | Mathlib4 (100K+ теорем) |
| Параллелизация | Ray, 24 процесса, 28 дней |
| Сгенерировано теорем | **4.7 миллиона** |
| Объём данных | **1 миллиард токенов** |
| Скорость | ~2035 состояний / теорему за 2 мин |
| Результат обучения | Flan-T5 (350M) побеждает ReProver на MiniF2F и MIL |

### Наша реализация в RE-RL

Мы воспроизводим ядро подхода LeanNavigator:

| Компонент | LeanNavigator | RE-RL |
|---|---|---|
| Взаимодействие с Lean 4 | LeanDojo | LeanDojo ✓ |
| BFS по графу состояний | ✓ | ✓ `StateExplorer` |
| Приоритетная очередь | ✓ | ✓ `PriorityQueue` |
| Rule-based тактики | ✓ | ✓ `TacticGenerator` |
| Шаблоны с подстановкой | ✓ | ✓ (гипотезы, переменные) |
| Сбор training pairs | ✓ | ✓ `TrainingPair` |
| Distance to proof | ✓ (≤8 шагов) | ✓ (backtrack) |
| Embedding retrieval (BERT+FAISS) | ✓ (ускорение) | ⚠️ Опционально (`ml_tactic_generator`) |
| Ray параллелизация | ✓ (24 процесса) | ❌ Планируется |
| Форматы экспорта | JSONL | JSONL / SFT / Chat ✓ |

---

## Что делает этот notebook

```
                        ┌──────────────┐
                        │ Lean Repo    │  (lean4-example, mathlib4, minif2f, ...)
                        │ (GitHub)     │
                        └──────┬───────┘
                               │
                    ┌──────────▼──────────┐
                    │  Tracing (LeanDojo) │  Компиляция + анализ AST (кэшируется)
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │  Извлечение теорем  │  get_traced_theorems()
                    └──────────┬──────────┘
                               │
              ┌────────────────▼────────────────┐
              │      BFS Exploration (ядро)      │
              │                                  │
              │  Для каждой теоремы:             │
              │  1. Открыть Dojo (Lean env)      │
              │  2. Генерировать тактики          │
              │  3. Применять → новые состояния  │
              │  4. Собирать (state, tactic) пары│
              └────────────────┬────────────────┘
                               │
                    ┌──────────▼──────────┐
                    │  Training Pairs     │  {state, tactic, next_state, distance}
                    └──────────┬──────────┘
                               │
              ┌────────────────▼────────────────┐
              │         Экспорт данных           │
              │  JSONL / SFT / Chat format      │
              └────────────────┬────────────────┘
                               │
                               ▼
                    Fine-tune LLM (Flan-T5, Qwen, ...)
                    → Theorem Prover
```

### Шаги notebook:
1. **Конфигурация** — выбор репозитория и параметров BFS
2. **Проверка зависимостей** — LeanDojo, elan, re_rl
3. **Tracing** — скачивание и компиляция Lean-репозитория (однократно, кэш `~/.cache/lean_dojo/`)
4. **Извлечение теорем** — из traced-репозитория
5. **BFS генерация** — исследование графа состояний, сбор training pairs
6. **Анализ** — статистика, примеры пар
7. **Сохранение** — JSONL/SFT/Chat для обучения LLM

---
# 1. Настройка окружения

In [ ]:
#@title Конфигурация {display-mode: "form"}

# === ВЫБЕРИТЕ РЕПОЗИТОРИЙ ===
# Доступные:
#   "lean4-example"          — 2 теоремы, тест pipeline за 1 минуту
#   "minif2f"                — 493 олимпиадные задачи (30-60 мин tracing)
#   "mathlib4"               — Mathlib4 (100K+ теорем, 3-6 часов tracing)
#   "mathematics-in-lean"    — учебник MIL (~500 теорем)
#
# re-rl использует собственный ExtractData.lean (из LeanDojo-v2),
# совместимый с Lean до v4.30.0. Можно использовать любой тег Mathlib4.
# Полный список тегов: https://github.com/leanprover-community/mathlib4/tags
REPO_NAME = "mathlib4"  #@param ["lean4-example", "minif2f", "mathlib4", "mathematics-in-lean"]

# Произвольная версия Mathlib4 (если заполнено — перебивает REPO_NAME).
CUSTOM_MATHLIB_VERSION = "v4.19.0"  # например "v4.19.0", "v4.28.0-rc1"

# === ПАРАМЕТРЫ ГЕНЕРАЦИИ ===
MAX_THEOREMS = 10           # Сколько теорем исследовать (0 = все)
MAX_STEPS_PER_THEOREM = 500  # Макс. шагов BFS на теорему
MAX_TIME_PER_THEOREM = 60    # Макс. секунд на теорему
MAX_DEPTH = 8               # Макс. глубина доказательства

# === ВЫВОД ===
OUTPUT_DIR = "./formal_math_data"  # Директория для датасета
OUTPUT_FORMAT = "jsonl"     #@param ["jsonl", "json", "sft", "chat"]

# Применяем custom версию если указана
if CUSTOM_MATHLIB_VERSION:
    from re_rl.tasks.formal import add_custom_mathlib4
    version = CUSTOM_MATHLIB_VERSION if CUSTOM_MATHLIB_VERSION.startswith("v") else f"v{CUSTOM_MATHLIB_VERSION}"
    REPO_NAME = add_custom_mathlib4(version)
    print(f"Используем Mathlib4 {version}")

print(f"Репозиторий: {REPO_NAME}")
print(f"Макс. теорем: {MAX_THEOREMS if MAX_THEOREMS > 0 else 'все'}")
print(f"Параметры BFS: steps={MAX_STEPS_PER_THEOREM}, time={MAX_TIME_PER_THEOREM}s, depth={MAX_DEPTH}")

Репозиторий: mathlib4
Макс. теорем: 10
Параметры BFS: steps=500, time=60s, depth=8


## 1.1 Проверка зависимостей

In [2]:
import subprocess
import sys
import os
from pathlib import Path

def check_dependencies():
    """Проверяет и показывает статус зависимостей."""
    print("=" * 60)
    print("ПРОВЕРКА ЗАВИСИМОСТЕЙ")
    print("=" * 60)
    
    all_ok = True
    
    # 1. LeanDojo
    try:
        import lean_dojo
        version = getattr(lean_dojo, '__version__', 'unknown')
        print(f"✓ LeanDojo: {version}")
    except ImportError:
        print("✗ LeanDojo: НЕ УСТАНОВЛЕН")
        print("  Установите: pip install lean-dojo")
        all_ok = False
    
    # 2. elan
    elan_path = Path.home() / ".elan" / "bin" / "elan"
    if elan_path.exists():
        print(f"✓ elan: установлен")
    else:
        print("✗ elan: НЕ УСТАНОВЛЕН")
        print("  Установите: curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh")
        all_ok = False
    
    # 3. re_rl
    try:
        import re_rl
        print(f"✓ re_rl: установлен")
    except ImportError:
        print("✗ re_rl: НЕ УСТАНОВЛЕН")
        print("  Установите: pip install -e .")
        all_ok = False
    
    print()
    
    if all_ok:
        print("✓ Все зависимости установлены!")
    else:
        print("⚠ Установите недостающие зависимости и перезапустите ячейку")
    
    return all_ok

DEPS_OK = check_dependencies()

/home/user/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-06 11:38:06,563	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


ПРОВЕРКА ЗАВИСИМОСТЕЙ
✓ LeanDojo: 4.20.0
✓ elan: установлен
✓ re_rl: установлен

✓ Все зависимости установлены!


In [3]:
# Установка зависимостей (раскомментируйте если нужно)

# !pip install lean-dojo
# !pip install -e ..  # Установить re_rl из родительской директории

# Для elan выполните в терминале:
# curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh

## 1.2 Импорты

In [4]:
# Добавляем elan в PATH
os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ.get("PATH", "")

import json
import time
from datetime import datetime
from typing import List, Dict, Any, Optional

# LeanDojo
from lean_dojo import LeanGitRepo, Theorem, Dojo, trace

# RE-RL formal math
from re_rl.tasks.formal import (
    StateExplorer,
    TacticGenerator,
    ExplorationResult,
    TrainingPair,
    LEAN_REPOS,
    trace_repo,
    list_cached_repos,
    _apply_extractor_fix,
)

# Подставляем наш исправленный ExtractData.lean (из LeanDojo-v2).
# Это позволяет трейсить Mathlib4 и другие репо на Lean до v4.30.0.
_apply_extractor_fix()

print("✓ Импорты загружены, ExtractData.lean подставлен")

✓ Импорты загружены


---
# 2. Подготовка репозитория

In [5]:
# Показываем доступные репозитории
print("ДОСТУПНЫЕ РЕПОЗИТОРИИ:")
print("=" * 60)
for name, info in LEAN_REPOS.items():
    cached = "[CACHED]" if any(name in r for r in list_cached_repos()) else ""
    print(f"  {name:25} | {info['theorems_count']:>8} теорем | {info['estimated_time']:>12} | {cached}")

print()
print(f"Выбран: {REPO_NAME}")

ДОСТУПНЫЕ РЕПОЗИТОРИИ:
  lean4-example             |        2 теорем |   1-2 минуты | [CACHED]
  minif2f                   |      493 теорем |  30-60 минут | 
  mathlib4                  |    100K+ теорем |    3-6 часов | 
  mathematics-in-lean       |     ~500 теорем |  30-60 минут | 

Выбран: mathlib4


In [6]:
# Загружаем информацию о репозитории
if REPO_NAME not in LEAN_REPOS:
    raise ValueError(f"Неизвестный репозиторий: {REPO_NAME}. Доступные: {list(LEAN_REPOS.keys())}")

repo_info = LEAN_REPOS[REPO_NAME]

print(f"Репозиторий: {REPO_NAME}")
print(f"  URL: {repo_info['url']}")
print(f"  Commit: {repo_info['commit']}")
print(f"  Описание: {repo_info['description']}")
print(f"  Теорем: {repo_info['theorems_count']}")
print(f"  Время tracing: {repo_info['estimated_time']}")

Репозиторий: mathlib4
  URL: https://github.com/leanprover-community/mathlib4
  Commit: v4.19.0
  Описание: Mathlib4 v4.19.0 (LeanDojo Benchmark 4, проверена) — 100K+ теорем
  Теорем: 100K+
  Время tracing: 3-6 часов


In [7]:
# Создаём LeanGitRepo и запускаем tracing
print("Создаём репозиторий...")
repo = LeanGitRepo(repo_info['url'], repo_info['commit'])

print("Запускаем tracing (при первом запуске скачиваются зависимости)...")
print("Это может занять время. Прогресс отображается ниже.")
print()

start_time = time.time()
traced_repo = trace(repo)
elapsed = time.time() - start_time

print()
print(f"✓ Tracing завершён за {elapsed:.1f}с ({elapsed/60:.1f} мин)")
print(f"  Файлов: {len(traced_repo.traced_files)}")

Создаём репозиторий...
Запускаем tracing (при первом запуске скачиваются зависимости)...
Это может занять время. Прогресс отображается ниже.



2026-02-06 11:38:08.979 | INFO     | lean_dojo.data_extraction.trace:get_traced_repo_path:210 - Tracing LeanGitRepo(url='https://github.com/leanprover-community/mathlib4', commit='c44e0c8ee63ca166450922a373c7409c5d26b00b')
info: downloading https://releases.lean-lang.org/lean4/v4.19.0/lean-4.19.0-linux.tar.zst
info: installing /home/user/.elan/toolchains/leanprover--lean4---v4.19.0
info: plausible: cloning https://github.com/leanprover-community/plausible
info: plausible: checking out revision '77e08eddc486491d7b9e470926b3dbe50319451a'
info: LeanSearchClient: cloning https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision '25078369972d295301f5a1e53c3e5850cf6d9d4c'
info: importGraph: cloning https://github.com/leanprover-community/import-graph
info: importGraph: checking out revision 'e6a9f0f5ee3ccf7443a0070f92b62f8db12ae82b'
info: proofwidgets: cloning https://github.com/leanprover-community/ProofWidgets4
info: proofwidgets: checking out r

✔ [3/6659] Built Mathlib.Tactic.Linter.DirectoryDependency
✔ [7/6659] Built Batteries.Data.List.ArrayMap
✔ [8/6659] Built Batteries.Util.Panic
✔ [9/6659] Built Mathlib.Tactic.Linter.Header
✔ [11/6659] Built Batteries.Control.Lemmas
✔ [12/6659] Built Batteries.Lean.System.IO
✔ [13/6659] Built Batteries.Lean.IO.Process
✔ [14/6659] Built Batteries.Lean.LawfulMonad
✔ [15/6659] Built Batteries.Util.Pickle
✔ [17/6659] Built Batteries.Data.BitVec.Basic
✔ [18/6659] Built Batteries.Data.ByteSubarray
✔ [19/6659] Built Batteries.Tactic.Lemma
✔ [20/6659] Built Batteries.Tactic.PrintOpaques
✔ [22/6659] Built Batteries.Lean.Except
✔ [23/6659] Built Batteries.WF
✔ [24/6659] Built Batteries.Lean.EStateM
✔ [25/6659] Built Batteries.Lean.Float
✔ [26/6659] Built Batteries.Data.Stream
✔ [27/6659] Built Batteries.Lean.Expr
✔ [28/6659] Built Batteries.Data.List.Init.Lemmas
✔ [29/6659] Built Batteries.Lean.PersistentHashMap
✔ [30/6659] Built Batteries.Data.NameSet
✔ [31/6659] Built Batteries.Lean.HashMap
✔ [

  0%|          | 0/8138 [00:00<?, ?it/s]

ExtractData.lean:479:41: error: application type mismatch
  getImports header
argument
  header
has type
  Syntax : Type
but is expected to have type
  TSyntax `Lean.Parser.Module.header : Type


CalledProcessError: Command 'lake env lean --threads 24 --run ExtractData.lean' returned non-zero exit status 1.

  0%|          | 0/8138 [1:00:17<?, ?it/s]

---
# 3. Получение списка теорем

In [ ]:
def get_theorems_from_repo(traced_repo, repo, max_count: int = 0) -> List[Theorem]:
    """
    Извлекает теоремы из traced репозитория.
    
    Args:
        traced_repo: Traced репозиторий от LeanDojo
        repo: LeanGitRepo объект
        max_count: Максимум теорем (0 = все)
    
    Returns:
        Список Theorem объектов
    """
    theorems = []
    
    for traced_file in traced_repo.traced_files:
        # Пропускаем файлы зависимостей
        file_path = str(traced_file.lean_file.path)
        if '.lake/packages' in file_path:
            continue
        
        try:
            # Получаем теоремы из файла
            for thm in traced_file.get_traced_theorems():
                theorem = Theorem(
                    repo=repo,
                    file_path=file_path,
                    full_name=thm.theorem.full_name
                )
                theorems.append(theorem)
                
                if max_count > 0 and len(theorems) >= max_count:
                    return theorems
        except Exception as e:
            # Некоторые файлы могут не содержать теорем
            continue
    
    return theorems

print("Извлекаем теоремы из репозитория...")
theorems = get_theorems_from_repo(traced_repo, repo, max_count=MAX_THEOREMS)

print(f"\n✓ Найдено теорем: {len(theorems)}")
print("\nПримеры:")
for thm in theorems[:5]:
    print(f"  - {thm.full_name} ({thm.file_path})")

Извлекаем теоремы из репозитория...

✓ Найдено теорем: 2

Примеры:
  - hello_world (Lean4Example.lean)
  - foo (Lean4Example.lean)


---
# 4. BFS Исследование (подход LeanNavigator)

In [ ]:
# Создаём StateExplorer с настройками
explorer = StateExplorer(
    max_steps=MAX_STEPS_PER_THEOREM,
    max_time=MAX_TIME_PER_THEOREM,
    max_depth=MAX_DEPTH,
    max_states=1000,
    verbose=False,
)

print("StateExplorer настроен:")
print(f"  max_steps: {explorer.max_steps}")
print(f"  max_time: {explorer.max_time}s")
print(f"  max_depth: {explorer.max_depth}")

StateExplorer настроен:
  max_steps: 500
  max_time: 60s
  max_depth: 8


In [ ]:
def explore_theorems(theorems: List[Theorem], explorer: StateExplorer) -> Dict[str, Any]:
    """
    Исследует список теорем и собирает training pairs.
    
    Returns:
        Словарь с результатами:
        - all_pairs: все training pairs
        - stats: статистика по каждой теореме
        - summary: общая статистика
    """
    all_pairs: List[TrainingPair] = []
    theorem_stats = []
    
    total_start = time.time()
    
    print(f"Исследуем {len(theorems)} теорем...")
    print("=" * 70)
    
    for i, theorem in enumerate(theorems):
        print(f"\n[{i+1}/{len(theorems)}] {theorem.full_name}")
        
        try:
            start = time.time()
            
            with Dojo(theorem) as (dojo, state_0):
                result, pairs, stats = explorer.explore(
                    dojo,
                    state_0,
                    theorem_name=theorem.full_name,
                    exit_on_proof=False,
                )
            
            elapsed = time.time() - start
            
            # Сохраняем статистику
            theorem_stat = {
                "theorem": theorem.full_name,
                "file": str(theorem.file_path),
                "result": result.value,
                "states": stats.total_states,
                "tactics_tried": stats.total_tactics_tried,
                "pairs": len(pairs),
                "proof_found": stats.proof_found,
                "max_depth": stats.max_depth_reached,
                "time": elapsed,
            }
            theorem_stats.append(theorem_stat)
            
            # Добавляем pairs
            all_pairs.extend(pairs)
            
            # Вывод
            status = "✓" if stats.proof_found else "○"
            print(f"  {status} states={stats.total_states}, pairs={len(pairs)}, "
                  f"proof={'yes' if stats.proof_found else 'no'}, time={elapsed:.1f}s")
            
        except Exception as e:
            print(f"  ✗ Ошибка: {str(e)[:50]}")
            theorem_stats.append({
                "theorem": theorem.full_name,
                "result": "error",
                "error": str(e),
            })
    
    total_time = time.time() - total_start
    
    # Общая статистика
    successful = [s for s in theorem_stats if s.get("result") == "success"]
    proofs_found = sum(1 for s in theorem_stats if s.get("proof_found", False))
    
    summary = {
        "total_theorems": len(theorems),
        "successful_explorations": len(successful),
        "proofs_found": proofs_found,
        "total_pairs": len(all_pairs),
        "total_time": total_time,
        "avg_time_per_theorem": total_time / len(theorems) if theorems else 0,
    }
    
    print("\n" + "=" * 70)
    print("ИТОГИ")
    print("=" * 70)
    print(f"  Теорем исследовано: {summary['total_theorems']}")
    print(f"  Доказательств найдено: {summary['proofs_found']}")
    print(f"  Training pairs: {summary['total_pairs']}")
    print(f"  Общее время: {summary['total_time']:.1f}s ({summary['total_time']/60:.1f} мин)")
    
    return {
        "all_pairs": all_pairs,
        "theorem_stats": theorem_stats,
        "summary": summary,
    }

# Запускаем исследование
results = explore_theorems(theorems, explorer)

Исследуем 2 теорем...

[1/2] hello_world
  ✓ states=51, pairs=60, proof=yes, time=39.2s

[2/2] foo


KeyboardInterrupt: 

---
# 5. Анализ результатов

In [ ]:
# Статистика по training pairs
all_pairs = results["all_pairs"]

print(f"АНАЛИЗ TRAINING PAIRS")
print("=" * 60)
print(f"Всего pairs: {len(all_pairs)}")

# Пары ведущие к доказательству
proof_pairs = [p for p in all_pairs if p.distance_to_proof >= 0]
print(f"Pairs на пути к ProofFinished: {len(proof_pairs)}")

# Уникальные тактики
unique_tactics = set(p.tactic for p in all_pairs)
print(f"Уникальных тактик: {len(unique_tactics)}")

# Топ тактики
print("\nТоп-10 тактик:")
tactic_counts = {}
for p in all_pairs:
    tactic_counts[p.tactic] = tactic_counts.get(p.tactic, 0) + 1

for tactic, count in sorted(tactic_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {tactic}: {count}")

АНАЛИЗ TRAINING PAIRS
Всего pairs: 846
Pairs на пути к ProofFinished: 754
Уникальных тактик: 33

Топ-10 тактик:
  intros: 355
  rfl: 84
  induction a with | zero => _ | succ n ih => _: 73
  have h := ih: 57
  specialize ih: 43
  specialize h: 42
  exfalso: 18
  cases a: 18
  congr: 15
  cases ih: 14


In [ ]:
# Примеры training pairs
print("ПРИМЕРЫ TRAINING PAIRS:")
print("=" * 60)

for i, pair in enumerate(proof_pairs[:5]):
    print(f"\n--- Pair {i+1} (distance={pair.distance_to_proof}) ---")
    print(f"State:")
    for line in pair.state.split('\n')[:5]:
        print(f"  {line}")
    print(f"Tactic: {pair.tactic}")
    print(f"Theorem: {pair.theorem_name}")

ПРИМЕРЫ TRAINING PAIRS:

--- Pair 1 (distance=0) ---
State:
  a b c : Nat
  ⊢ a + b + c = a + c + b
Tactic: omega
Theorem: hello_world

--- Pair 2 (distance=1) ---
State:
  a b c : Nat
  ⊢ a + b + c = a + c + b
Tactic: norm_cast
Theorem: hello_world

--- Pair 3 (distance=0) ---
State:
  a : Nat
  ⊢ a + 1 = a.succ
Tactic: rfl
Theorem: foo

--- Pair 4 (distance=1) ---
State:
  a : Nat
  ⊢ a + 1 = a.succ
Tactic: intros
Theorem: foo

--- Pair 5 (distance=0) ---
State:
  a : Nat
  ⊢ a + 1 = a.succ
Tactic: simp
Theorem: foo


---
# 6. Сохранение датасета

In [ ]:
def save_dataset(pairs: List[TrainingPair], output_dir: str, format: str, metadata: Dict):
    """
    Сохраняет датасет в указанном формате.
    
    Formats:
        - jsonl: JSON Lines (один объект на строку)
        - json: Один JSON файл
        - sft: Формат для Supervised Fine-Tuning
        - chat: Chat формат для conversational models
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if format == "jsonl":
        filename = output_path / f"lean_data_{timestamp}.jsonl"
        with open(filename, "w") as f:
            for pair in pairs:
                record = {
                    "state": pair.state,
                    "tactic": pair.tactic,
                    "next_state": pair.next_state,
                    "distance_to_proof": pair.distance_to_proof,
                    "theorem_name": pair.theorem_name,
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    elif format == "json":
        filename = output_path / f"lean_data_{timestamp}.json"
        data = [pair.to_dict() for pair in pairs]
        with open(filename, "w") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    
    elif format == "sft":
        filename = output_path / f"lean_sft_{timestamp}.json"
        sft_data = []
        for pair in pairs:
            sft_data.append({
                "instruction": "You are a Lean 4 theorem prover. Given the current proof state, suggest the next tactic to apply.",
                "input": f"Current proof state:\n{pair.state}",
                "output": pair.tactic,
            })
        with open(filename, "w") as f:
            json.dump(sft_data, f, indent=2, ensure_ascii=False)
    
    elif format == "chat":
        filename = output_path / f"lean_chat_{timestamp}.json"
        chat_data = []
        for pair in pairs:
            chat_data.append({
                "messages": [
                    {"role": "system", "content": "You are an expert Lean 4 theorem prover. Given a proof state, suggest the best tactic to make progress toward the proof."},
                    {"role": "user", "content": f"Prove this goal:\n```\n{pair.state}\n```"},
                    {"role": "assistant", "content": pair.tactic},
                ]
            })
        with open(filename, "w") as f:
            json.dump(chat_data, f, indent=2, ensure_ascii=False)
    
    else:
        raise ValueError(f"Unknown format: {format}")
    
    # Сохраняем метаданные
    meta_filename = output_path / f"metadata_{timestamp}.json"
    with open(meta_filename, "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"✓ Сохранено: {filename}")
    print(f"✓ Метаданные: {meta_filename}")
    
    return filename

# Подготавливаем метаданные
metadata = {
    "repo_name": REPO_NAME,
    "repo_url": repo_info['url'],
    "repo_commit": repo_info['commit'],
    "generation_date": datetime.now().isoformat(),
    "config": {
        "max_theorems": MAX_THEOREMS,
        "max_steps_per_theorem": MAX_STEPS_PER_THEOREM,
        "max_time_per_theorem": MAX_TIME_PER_THEOREM,
        "max_depth": MAX_DEPTH,
    },
    "summary": results["summary"],
}

# Сохраняем
output_file = save_dataset(all_pairs, OUTPUT_DIR, OUTPUT_FORMAT, metadata)

✓ Сохранено: formal_math_data/lean_data_20260206_105420.jsonl
✓ Метаданные: formal_math_data/metadata_20260206_105420.json


In [ ]:
# Показываем содержимое
print(f"\nСодержимое {OUTPUT_DIR}:")
for f in Path(OUTPUT_DIR).iterdir():
    size = f.stat().st_size
    print(f"  {f.name}: {size:,} bytes")

print(f"\nПервые строки датасета:")
with open(output_file) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line[:200] + "..." if len(line) > 200 else line)


Содержимое ./formal_math_data:
  lean_data_20260206_105420.jsonl: 210,235 bytes
  metadata_20260206_105420.json: 541 bytes

Первые строки датасета:
{"state": "a b c : Nat\n⊢ a + b + c = a + c + b", "tactic": "omega", "next_state": "ProofFinished", "distance_to_proof": 0, "theorem_name": "hello_world"}

{"state": "a b c : Nat\n⊢ a + b + c = a + c + b", "tactic": "norm_cast", "next_state": "a b c : Nat\n⊢ a + b + c = a + c + b", "distance_to_proof": 1, "theorem_name": "hello_world"}

{"state": "a b c : Nat\n⊢ a + b + c = a + c + b", "tactic": "congr", "next_state": "case e_a.e_a\na b c : Nat\n⊢ b = c\n\ncase e_a\na b c : Nat\n⊢ c = b", "distance_to_proof": -1, "theorem_name": "hel...


---
# 7. Следующие шаги

Теперь у вас есть датасет для обучения theorem prover!

## Варианты использования:

### 1. Fine-tune LLM (SFT)
```python
from transformers import AutoModelForCausalLM, Trainer

# Загрузите lean_sft_*.json и обучите модель
```

### 2. GRPO с верификацией
```python
# Используйте Lean для верификации как reward signal
# reward = 1.0 if proof_verified else 0.0
```

### 3. Масштабирование
- Используйте `mathlib4` для 100K+ теорем
- Добавьте Ray параллелизацию для ускорения

## Полезные ссылки:
- [LeanNavigator Paper](https://arxiv.org/abs/...)
- [LeanDojo Documentation](https://leandojo.readthedocs.io/)
- [Mathlib4](https://github.com/leanprover-community/mathlib4)

In [ ]:
print("=" * 60)
print("ГОТОВО!")
print("=" * 60)
print(f"\nСгенерировано {len(all_pairs)} training pairs из {len(theorems)} теорем.")
print(f"Датасет сохранён в: {OUTPUT_DIR}")
print(f"\nЭто воспроизводимый пайплайн как в LeanNavigator!")

ГОТОВО!

Сгенерировано 846 training pairs из 2 теорем.
Датасет сохранён в: ./formal_math_data

Это воспроизводимый пайплайн как в LeanNavigator!
